<a href="https://colab.research.google.com/github/skyexry/urban-mobility-forecast/blob/main/notebooks/04_model_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/skyexry/urban-mobility-forecast.git

fatal: destination path 'urban-mobility-forecast' already exists and is not an empty directory.


In [ ]:
import sys
sys.path.append('/content/urban-mobility-forecast')

In [ ]:
!git -C /content/urban-mobility-forecast pull

Already up to date.


In [ ]:
import torch
from model.tcn import TCNBlock
from model.stconv import STConvBlock, build_laplacian
from model.stgnn import STGNN

## 1. TCN Block


In [ ]:
# Sanity check
batch, time, features = 16, 72, 6
x = torch.randn(batch, features, time)

tcn = TCNBlock(in_channels=features, out_channels=64)
out = tcn(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")

Input:  torch.Size([16, 6, 72])
Output: torch.Size([16, 64, 72])


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


## 2. STCONV

In [ ]:
# Sanity check
batch, N, T = 4, 10, 72

x = torch.randn(batch, 1, N, T)
W = torch.rand(N, N)
W = (W + W.T) / 2
L_hat = build_laplacian(W)

block = STConvBlock(in_channels=1, hidden_channels=16, out_channels=32, kernel_size=3, K=3)
out = block(x, L_hat)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")  # (4, 32, 10, 68)

Input:  torch.Size([4, 1, 10, 72])
Output: torch.Size([4, 32, 10, 68])


## 3. STGNN

In [ ]:
# Sanity check with small N first
batch, N, T = 4, 20, 72

x_demand = torch.randn(batch, N, T, 1)
x_time = torch.randn(batch, T, 6)

# Build Laplacian from random adjacency matrix
W = torch.rand(N, N)
W = (W + W.T) / 2
L_hat = build_laplacian(W)

model = STGNN(num_nodes=N)
y_hat = model(x_demand, x_time, L_hat)
print(f"x_demand: {x_demand.shape}")
print(f"x_time:   {x_time.shape}")
print(f"y_hat:    {y_hat.shape}")   # 应该是 (4, 20, 72)

x_demand: torch.Size([4, 20, 72, 1])
x_time:   torch.Size([4, 72, 6])
y_hat:    torch.Size([4, 20, 72])
